# Traditional ANN (v1)

In [75]:
import math

import norse
import torch
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all" # type: ignore

# torch.manual_seed(1337)
N = 1000
X = torch.randint(0, 2, (N, 2)).float()
Y = (X[:, 0] != X[:, 1]).long()
split = math.floor(N * 0.9)
X_train = X[:split]
Y_train = Y[:split]
X_test = X[split:]
Y_test = Y[split:]

model = torch.nn.Sequential(
    torch.nn.Linear(2, 10),
    torch.nn.ReLU(),
    torch.nn.Linear(10, 2),
)

loss_fn = torch.nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-2)

for epoch in range(100):
    opt.zero_grad()
    loss = loss_fn(model(X_train), Y_train)
    loss.backward()
    opt.step()

Y_hat = model(X_test)
acc = (Y_hat.argmax(1) == Y_test).float().mean()
print(torch.column_stack((X_test, Y_test, Y_hat)))
print(acc)
# acc = (model(X_test).argmax(1) == Y_test).float().mean()
# acc



tensor([[ 0.0000,  0.0000,  0.0000,  1.3085, -1.1121],
        [ 1.0000,  1.0000,  0.0000,  0.9429, -1.2942],
        [ 1.0000,  0.0000,  1.0000, -0.8745,  1.3546],
        [ 1.0000,  0.0000,  1.0000, -0.8745,  1.3546],
        [ 1.0000,  0.0000,  1.0000, -0.8745,  1.3546],
        [ 0.0000,  1.0000,  1.0000, -1.1663,  1.2420],
        [ 0.0000,  1.0000,  1.0000, -1.1663,  1.2420],
        [ 1.0000,  0.0000,  1.0000, -0.8745,  1.3546],
        [ 1.0000,  0.0000,  1.0000, -0.8745,  1.3546],
        [ 0.0000,  1.0000,  1.0000, -1.1663,  1.2420],
        [ 1.0000,  1.0000,  0.0000,  0.9429, -1.2942],
        [ 0.0000,  0.0000,  0.0000,  1.3085, -1.1121],
        [ 1.0000,  0.0000,  1.0000, -0.8745,  1.3546],
        [ 0.0000,  1.0000,  1.0000, -1.1663,  1.2420],
        [ 1.0000,  1.0000,  0.0000,  0.9429, -1.2942],
        [ 1.0000,  0.0000,  1.0000, -0.8745,  1.3546],
        [ 0.0000,  1.0000,  1.0000, -1.1663,  1.2420],
        [ 1.0000,  1.0000,  0.0000,  0.9429, -1.2942],
        [ 

# Traditional ANN (v2)

In [56]:
torch.manual_seed(1337)

# XOR is just 4 patterns: train on the full truth table.
X = torch.tensor([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]) # 4 x 2
Y = (X[:, 0] != X[:, 1]).float().unsqueeze(1) # 4 x 1
H = 8
model = torch.nn.Sequential(
    torch.nn.Linear(2, H),
    torch.nn.Tanh(),
    torch.nn.Linear(H, 1),  # single logit: P(y=1) = sigmoid(logit)
)

loss_fn = torch.nn.BCEWithLogitsLoss()  # sigmoid + binary cross-entropy, as in Payeur's cost
opt = torch.optim.Adam(model.parameters(), lr=1e-2)

for epoch in range(500):
    opt.zero_grad()
    loss = loss_fn(model(X), Y)
    loss.backward()
    opt.step()

with torch.no_grad():
    p = torch.sigmoid(model(X))
acc = ((p > 0.5).float() == Y).float().mean()
print(f"loss = {loss.item():.4f}, acc = {acc.item():.2f}")
print(torch.column_stack((X, Y, p)))

loss = 0.0036, acc = 1.00
tensor([[0.0000, 0.0000, 0.0000, 0.0043],
        [1.0000, 0.0000, 1.0000, 0.9980],
        [0.0000, 1.0000, 1.0000, 0.9976],
        [1.0000, 1.0000, 0.0000, 0.0058]])


# SNN

In [ ]:

# Constant current input.
X = torch.tensor([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
Y = (X[:, 0] != X[:, 1]).float().unsqueeze(1) # 4 x 1
T = 100
# Hold the input constant for T timesteps.
X_seq = X.expand(T, *X.shape) # (T, 4, 2)

model = norse.torch.SequentialState(
    torch.nn.Linear(2, 10),
    norse.torch.LIF(),
    torch.nn.Linear(10, 1),
    norse.torch.LI()
)
loss_fn = torch.nn.BCEWithLogitsLoss()
opt = torch.optim.Adam(model.parameters(), lr=2e-3)

# states
for epoch in range(1000):
    opt.zero_grad()
    out, states = model(X_seq)
    loss = loss_fn(out[-1], Y)
    print(f"Loss={loss}")
    loss.backward()
    opt.step()

Loss=0.7359122633934021
Loss=0.7304859161376953
Loss=0.7284858226776123
Loss=0.7283904552459717
Loss=0.7144920825958252
Loss=0.7138388752937317
Loss=0.6976462006568909
Loss=0.6947036385536194
Loss=0.6899110078811646
Loss=0.6842745542526245
Loss=0.6827410459518433
Loss=0.6817320585250854
Loss=0.6764243245124817
Loss=0.6751126646995544
Loss=0.6755002737045288
Loss=0.6670618057250977
Loss=0.6684941053390503
Loss=0.6679774522781372
Loss=0.6657670140266418
Loss=0.6636679172515869
Loss=0.6672429442405701
Loss=0.6573038101196289
Loss=0.6610198020935059
Loss=0.6482862234115601
Loss=0.651196300983429
Loss=0.6508625149726868
Loss=0.6384600400924683
Loss=0.638187825679779
Loss=0.6398382186889648
Loss=0.6403430700302124
Loss=0.6391865015029907
Loss=0.6320922374725342
Loss=0.6322075128555298
Loss=0.6326711177825928
Loss=0.6361201405525208
Loss=0.6383009552955627
Loss=0.6382154822349548
Loss=0.6367151737213135
Loss=0.6314963698387146
Loss=0.6321967840194702
Loss=0.6295758485794067
Loss=0.63145262002